# UPI / Fintech Transaction Analysis — 2023
### Author: Shubham Jadhav | Data Analyst Portfolio Project

---

**Project Summary:**  
Analysis of 100,000 UPI transactions across India in 2023.  
Key focus: transaction trends, payment method performance, fraud detection, user segmentation.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F9FAFB",
    "axes.grid": True,
    "grid.color": "#E5E7EB",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, GREEN, RED, ORANGE, PURPLE = "#2563EB", "#10B981", "#EF4444", "#F59E0B", "#8B5CF6"
PALETTE = [BLUE, GREEN, RED, ORANGE, PURPLE, "#14B8A6", "#F97316", "#6366F1", "#84CC16", "#EC4899"]

print("Libraries loaded!")

In [ ]:
df = pd.read_csv("../data/upi_transactions.csv", parse_dates=["timestamp"])

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")
print(f"Date range: {df["timestamp"].min().date()} to {df["timestamp"].max().date()}")

In [ ]:
df.head()

In [ ]:
print(df.dtypes)
print("
Missing values:")
print(df.isnull().sum())

In [ ]:
df[["amount"]].describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
print(df["status"].value_counts())
print(f"
Success Rate: {(df["status"]=="Success").mean()*100:.2f}%")

In [ ]:
success_df = df[df["status"] == "Success"].copy()
print(f"Successful transactions: {len(success_df):,}")
print(f"Total value: ₹{success_df["amount"].sum()/1e7:.2f} Crore")

## 3. Monthly Transaction Trends
Is the platform growing month over month?

In [ ]:
monthly = success_df.groupby("month").agg(
    transaction_count = ("transaction_id", "count"),
    total_value       = ("amount", "sum"),
    avg_value         = ("amount", "mean")
).reset_index()

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
monthly["month_name"] = monthly["month"].apply(lambda x: month_names[x-1])
print(monthly[["month_name","transaction_count","total_value","avg_value"]].to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(monthly["month_name"], monthly["transaction_count"], color=BLUE, alpha=0.75, label="Transaction Count")
ax2.plot(monthly["month_name"], monthly["total_value"]/1e6, color=RED, marker="o", linewidth=2.5, label="Total Value (M)")

ax1.set_ylabel("Number of Transactions", color=BLUE)
ax2.set_ylabel("Total Value (Millions)", color=RED)
ax1.tick_params(axis="y", labelcolor=BLUE)
ax2.tick_params(axis="y", labelcolor=RED)
plt.title("Monthly UPI Transaction Volume & Value — 2023", fontsize=14, fontweight="bold")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc="upper left")
plt.tight_layout()
plt.show()
print("Insight: Consistent volume throughout 2023, slight festive peaks in Oct-Nov.")

## 4. Payment Method Analysis

In [ ]:
pm_count = success_df["payment_method"].value_counts()
pm_value = success_df.groupby("payment_method")["amount"].sum().sort_values(ascending=False)
for app, cnt in pm_count.items():
    print(f"{app:<12}: {cnt:>6,}  ({cnt/len(success_df)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].pie(pm_count.values, labels=pm_count.index, autopct="%1.1f%%", colors=PALETTE, startangle=140, wedgeprops={"edgecolor":"white","linewidth":2})
axes[0].set_title("Transaction Count Share", fontweight="bold")
axes[1].pie(pm_value.values, labels=pm_value.index, autopct="%1.1f%%", colors=PALETTE, startangle=140, wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Transaction Value Share", fontweight="bold")
fig.suptitle("Payment Method Market Share", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Peak Hour Heatmap

In [ ]:
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
heatmap_data = success_df.groupby(["day_of_week","hour"]).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(day_order)
print(f"Peak hour: {success_df["hour"].value_counts().idxmax()}:00")
print(f"Busiest day: {success_df["day_of_week"].value_counts().idxmax()}")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap="Blues", ax=ax, linewidths=0.3, linecolor="white", cbar_kws={"label": "Transactions"})
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Day of Week")
ax.set_title("Transaction Volume Heatmap — Peak Hours by Day", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print("Insight: Peak activity 10AM-2PM weekdays. Weekend evenings 6-9PM also high.")

## 6. Category Analysis

In [ ]:
cat_stats = success_df.groupby("category").agg(
    total_value=("amount","sum"), count=("amount","count"), avg_value=("amount","mean")
).sort_values("total_value", ascending=False).round(2)
print(cat_stats.to_string())

In [ ]:
cat_plot = cat_stats.sort_values("total_value", ascending=True)
colors = [PALETTE[i % len(PALETTE)] for i in range(len(cat_plot))]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(cat_plot.index, cat_plot["total_value"]/1e6, color=colors, edgecolor="white")
axes[0].set_xlabel("Total Value (Millions)")
axes[0].set_title("Total Spend by Category", fontweight="bold")
axes[1].barh(cat_plot.index, cat_plot["avg_value"], color=colors, edgecolor="white")
axes[1].set_xlabel("Average Transaction Value")
axes[1].set_title("Avg Transaction Value by Category", fontweight="bold")
fig.suptitle("Spending Analysis by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Fraud Detection Analysis
Most critical section for fintech hiring managers.

In [ ]:
total_fraud = df["is_fraud"].sum()
fraud_rate  = df["is_fraud"].mean() * 100
fraud_value = df[df["is_fraud"]==1]["amount"].sum()
print(f"Total flagged: {total_fraud:,}")
print(f"Fraud rate: {fraud_rate:.2f}%")
print(f"Fraud value: {fraud_value/1e5:.2f} Lakh")
print(df.groupby("payment_method")["is_fraud"].mean().mul(100).round(2).sort_values(ascending=False))

In [ ]:
df["amount_bucket"] = pd.cut(df["amount"], bins=[0,500,2000,10000,50000,999999], labels=["0-500","500-2K","2K-10K","10K-50K","50K+"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fraud_pm = df.groupby("payment_method")["is_fraud"].mean().mul(100).sort_values(ascending=False).reset_index()
axes[0].bar(fraud_pm["payment_method"], fraud_pm["is_fraud"], color=RED, alpha=0.8)
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_title("Fraud by Payment Method", fontweight="bold")
axes[0].tick_params(axis="x", rotation=20)

fraud_amt = df.groupby("amount_bucket", observed=True)["is_fraud"].mean().mul(100)
axes[1].bar(fraud_amt.index, fraud_amt.values, color=ORANGE, alpha=0.85)
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_title("Fraud by Transaction Size", fontweight="bold")

fraud_hour = df.groupby("hour")["is_fraud"].mean().mul(100)
axes[2].plot(fraud_hour.index, fraud_hour.values, color=RED, linewidth=2.5, marker="o", markersize=4)
axes[2].fill_between(fraud_hour.index, fraud_hour.values, alpha=0.15, color=RED)
axes[2].set_xlabel("Hour of Day")
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].set_title("Fraud by Hour of Day", fontweight="bold")

fig.suptitle("Fraud Detection Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print("Insight: Transactions >50K have 8x higher fraud. Late-night hours also elevated.")
print("Recommendation: Mandatory 2FA for transactions >10,000 between 1AM-4AM.")

## 8. User Segmentation

In [ ]:
user_spend = success_df.groupby("user_id").agg(
    total_spend=("amount","sum"), txn_count=("amount","count"), avg_txn=("amount","mean")
).reset_index()

user_spend["segment"] = pd.cut(
    user_spend["total_spend"],
    bins=[0, 1000, 10000, 50000, 9999999],
    labels=["Low Value", "Medium Value", "High Value", "VIP"]
)
print(user_spend["segment"].value_counts().sort_index())
print(f"
Avg annual spend per user: {user_spend["total_spend"].mean():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

seg_counts = user_spend["segment"].value_counts().sort_index()
axes[0].bar(seg_counts.index, seg_counts.values, color=[GREEN, BLUE, ORANGE, PURPLE], edgecolor="white", alpha=0.85)
axes[0].set_ylabel("Number of Users")
axes[0].set_title("User Segmentation by Annual Spend", fontweight="bold")
for i, v in enumerate(seg_counts.values):
    axes[0].text(i, v+20, f"{v:,}", ha="center", fontsize=10, fontweight="bold")

city_stats = success_df.groupby("city")["amount"].sum().sort_values(ascending=False)
axes[1].bar(city_stats.index, city_stats.values/1e6, color=PALETTE[:len(city_stats)], edgecolor="white", alpha=0.85)
axes[1].set_ylabel("Total Value (Millions)")
axes[1].set_title("Transaction Value by City", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)

fig.suptitle("User & Geographic Segmentation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Key Findings & Business Recommendations

### Summary
| Metric | Value |
|--------|-------|
| Total Transactions | 1,00,000 |
| Success Rate | 94.0% |
| Total Value | ₹4.02 Crore |
| Avg Transaction | ₹427.71 |
| Fraud Rate | 0.88% |
| Unique Users | ~15,000 |
| Top App | GPay (35%) |
| Peak Hour | 10:00 AM |

### Business Recommendations
1. **Fraud Prevention**: Transactions >₹10K between 1AM–4AM show highest fraud. Implement mandatory 2FA.
2. **User Retention**: 80% users are Low/Medium value. Cashback loyalty program could upgrade them.
3. **Infrastructure**: Pre-provision 30% extra capacity at 10AM–2PM on weekdays.
4. **Category Strategy**: Education & Transfer have highest ticket size — build premium features for them.
5. **City Focus**: Delhi & Mumbai top cities — hyperlocal merchant partnerships would have highest ROI.